In [2]:
import sys
!{sys.executable} -m pip install Faker


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime, timedelta
from pathlib import Path

print("All imports worked")

All imports worked


In [9]:
# ----------------------------
# Reproducibility
# ----------------------------
random.seed(42)
np.random.seed(42)
Faker.seed(42)
fake = Faker()

# ----------------------------
# Project folders
# ----------------------------
base_path = Path.cwd()  # notebook inside /notebooks, project root is one level up
data_path = base_path / "data"
output_path = base_path / "output"

data_path.mkdir(exist_ok=True)
output_path.mkdir(exist_ok=True)

# ----------------------------
# Business configuration
# ----------------------------
channels = ["Website", "Amazon", "Flipkart", "Instagram Shop", "Myntra", "Nykaa"]

products = [
    {"sku": "LIP001", "product_name": "Velvet Matte Lipstick", "category": "Lips", "unit_price": 699},
    {"sku": "LIP002", "product_name": "Hydra Gloss Tint", "category": "Lips", "unit_price": 599},
    {"sku": "EYE001", "product_name": "Smudgeproof Kajal", "category": "Eyes", "unit_price": 349},
    {"sku": "FACE001", "product_name": "Radiance Compact", "category": "Face", "unit_price": 799},
    {"sku": "FACE002", "product_name": "Skin Blur Primer", "category": "Face", "unit_price": 899},
    {"sku": "BASE001", "product_name": "Silk Finish Foundation", "category": "Base", "unit_price": 999},
    {"sku": "FACE003", "product_name": "Cream Blush", "category": "Face", "unit_price": 649},
    {"sku": "SKIN001", "product_name": "Dew Fix Setting Spray", "category": "Skincare", "unit_price": 749},
]

start_date = datetime(2026, 2, 1)
end_date   = datetime(2026, 5, 18)

# ----------------------------
# Helper function
# ----------------------------
def random_customer_type():
    return random.choices(
        ["New", "Returning"],
        weights=[0.4, 0.6],
        k=1
    )[0]

def random_channel():
    return random.choices(
        channels,
        weights=[22, 24, 16, 10, 12, 16],
        k=1
    )[0]

def seasonal_multiplier(date_value):
    # Weekend boost
    weekend_boost = 1.15 if date_value.weekday() >= 5 else 1.0
    
    # Small month-start / month-end spike
    payday_boost = 1.10 if date_value.day in [1, 2, 3, 28, 29, 30, 31] else 1.0
    
    # Occasional promotion spike
    promo_boost = 1.50 if random.random() < 0.05 else 1.0
    
    return weekend_boost * payday_boost * promo_boost

print("Configuration ready.")
print("Data folder:", data_path)
print("Output folder:", output_path)

Configuration ready.
Data folder: C:\Users\svans\weekly_beauty_report_project\data
Output folder: C:\Users\svans\weekly_beauty_report_project\output


In [10]:
rows = []

current_date = start_date
order_counter = 10001

while current_date <= end_date:
    day_factor = seasonal_multiplier(current_date)
    
    # Each day will have multiple sales rows
    transactions_today = random.randint(18, 35)
    
    for _ in range(transactions_today):
        product = random.choice(products)
        channel = random_channel()
        customer_type = random_customer_type()
        
        units_sold = max(1, int(np.random.normal(loc=4 * day_factor, scale=2)))
        orders = max(1, int(round(units_sold * random.uniform(0.7, 1.0))))
        
        gross_revenue = units_sold * product["unit_price"]
        discount_amount = round(gross_revenue * random.uniform(0.03, 0.18), 2)
        net_revenue = round(gross_revenue - discount_amount, 2)
        
        returns = np.random.binomial(units_sold, 0.03)
        
        rows.append({
            "order_id": f"ORD{order_counter}",
            "date": current_date.strftime("%Y-%m-%d"),
            "channel": channel,
            "sku": product["sku"],
            "product_name": product["product_name"],
            "category": product["category"],
            "customer_type": customer_type,
            "orders": orders,
            "units_sold": units_sold,
            "gross_revenue": round(gross_revenue, 2),
            "discount_amount": discount_amount,
            "net_revenue": net_revenue,
            "returns": returns
        })
        
        order_counter += 1
    
    current_date += timedelta(days=1)

df_clean = pd.DataFrame(rows)

print("Clean dataset created successfully.")
print("Shape:", df_clean.shape)
display(df_clean.head())

Clean dataset created successfully.
Shape: (2865, 13)


,order_id,date,channel,sku,product_name,category,customer_type,orders,units_sold,gross_revenue,discount_amount,net_revenue,returns
0,ORD10001,2026-02-01,Amazon,FACE002,Skin Blur Primer,Face,New,4,6,5394,761.09,4632.91,0
1,ORD10002,2026-02-01,Flipkart,LIP002,Hydra Gloss Tint,Lips,New,3,4,2396,155.50,2240.50,0
2,ORD10003,2026-02-01,Flipkart,LIP001,Velvet Matte Lipstick,Lips,Returning,4,4,2796,259.83,2536.17,0
3,ORD10004,2026-02-01,Flipkart,SKIN001,Dew Fix Setting Spray,Skincare,Returning,3,4,2996,452.02,2543.98,0
4,ORD10005,2026-02-01,Amazon,FACE003,Cream Blush,Face,New,8,8,5192,417.90,4774.10,0


In [11]:
df_dirty = df_clean.copy()

# ----------------------------
# Inject dirty problems
# ----------------------------

# 1. Duplicate rows
duplicate_rows = df_dirty.sample(12, random_state=42)
df_dirty = pd.concat([df_dirty, duplicate_rows], ignore_index=True)

# 2. Inconsistent channel labels
dirty_idx = df_dirty.sample(9, random_state=7).index
df_dirty.loc[dirty_idx[:3], "channel"] = "instagram shop"
df_dirty.loc[dirty_idx[3:6], "channel"] = "AMAZON "
df_dirty.loc[dirty_idx[6:], "channel"] = "Nykaa "

# 3. Missing values in net_revenue
missing_idx = df_dirty.sample(6, random_state=15).index
df_dirty.loc[missing_idx[:3], "net_revenue"] = np.nan

# 4. Invalid negative orders
df_dirty.loc[missing_idx[3:5], "orders"] = -1

# 5. Bad date format
df_dirty.loc[missing_idx[5:], "date"] = "18-05-2026"

# ----------------------------
# Data quality checks
# ----------------------------
print("Dirty dataset created.")
print("Shape:", df_dirty.shape)

print("\nMissing values per column:")
print(df_dirty.isna().sum())

print("\nDuplicate row count:")
print(df_dirty.duplicated().sum())

print("\nUnique channel values:")
print(df_dirty["channel"].unique())

print("\nRows with negative orders:")
print((df_dirty["orders"] < 0).sum())

display(df_dirty.head())

Dirty dataset created.
Shape: (2877, 13)

Missing values per column:
order_id           0
date               0
channel            0
sku                0
product_name       0
category           0
customer_type      0
orders             0
units_sold         0
gross_revenue      0
discount_amount    0
net_revenue        3
returns            0
dtype: int64

Duplicate row count:
12

Unique channel values:
['Amazon' 'Flipkart' 'Myntra' 'Website' 'Nykaa' 'Instagram Shop' 'AMAZON '
 'Nykaa ' 'instagram shop']

Rows with negative orders:
2


,order_id,date,channel,sku,product_name,category,customer_type,orders,units_sold,gross_revenue,discount_amount,net_revenue,returns
0,ORD10001,2026-02-01,Amazon,FACE002,Skin Blur Primer,Face,New,4,6,5394,761.09,4632.91,0
1,ORD10002,2026-02-01,Flipkart,LIP002,Hydra Gloss Tint,Lips,New,3,4,2396,155.50,2240.50,0
2,ORD10003,2026-02-01,Flipkart,LIP001,Velvet Matte Lipstick,Lips,Returning,4,4,2796,259.83,2536.17,0
3,ORD10004,2026-02-01,Flipkart,SKIN001,Dew Fix Setting Spray,Skincare,Returning,3,4,2996,452.02,2543.98,0
4,ORD10005,2026-02-01,Amazon,FACE003,Cream Blush,Face,New,8,8,5192,417.90,4774.10,0


In [12]:
clean_file_path = data_path / "beauty_sales_clean.csv"
dirty_file_path = data_path / "beauty_sales_dirty.csv"

df_clean.to_csv(clean_file_path, index=False)
df_dirty.to_csv(dirty_file_path, index=False)

print("Both files saved successfully.")
print("Clean file saved at:", clean_file_path)
print("Dirty file saved at:", dirty_file_path)

Both files saved successfully.
Clean file saved at: C:\Users\svans\weekly_beauty_report_project\data\beauty_sales_clean.csv
Dirty file saved at: C:\Users\svans\weekly_beauty_report_project\data\beauty_sales_dirty.csv


In [13]:
from pathlib import Path
print(Path.cwd())

C:\Users\svans\weekly_beauty_report_project
